# 01 — STEAM_RUN_EDA (Databricks + PySpark)

**Goal:** Load Steam dataset (nested JSON), build a clean `games` table, normalize nested fields (genres, languages, platforms), and produce dashboard-ready aggregation tables using `display()`.

Dataset: `s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json`


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

PATH = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"


In [ ]:
raw = spark.read.json(PATH)

raw.printSchema()
display(raw.limit(20))
print("rows =", raw.count(), "| columns =", len(raw.columns))
print("top-level columns:", raw.columns)


## Helpers (safe access + cleaning)

In [ ]:
def has_path(schema, path: str) -> bool:
    """Return True if a nested path exists in schema (e.g. 'data.platforms.windows')."""
    parts = path.split(".")
    dt = schema
    for p in parts:
        if isinstance(dt, T.StructType):
            f = next((x for x in dt.fields if x.name == p), None)
            if f is None:
                return False
            dt = f.dataType
        else:
            return False
    return True

def col_or_null(path: str):
    return F.col(path) if has_path(raw.schema, path) else F.lit(None)

def to_double_from_any(c):
    """Extract numeric value from string/number and cast to double. Handles comma decimals."""
    return F.regexp_replace(
        F.regexp_extract(c.cast("string"), r"([0-9]+[.,]?[0-9]*)", 1),
        ",", "."
    ).cast("double")

def pct_from_any(c):
    """Convert percent/ratio strings to a percentage double (0-100)."""
    x = to_double_from_any(c)
    return F.when(c.isNull(), F.lit(None)) \
            .when(x <= 1, x * 100) \
            .otherwise(x)


## Build clean `games` table (gold)

In [ ]:
games = (
    raw
    .withColumn("app_id", F.coalesce(col_or_null("data.appid"), col_or_null("id")).cast("long"))
    .withColumn("name", col_or_null("data.name").cast("string"))
    .withColumn("developer", col_or_null("data.developer").cast("string"))
    .withColumn("publisher", F.coalesce(col_or_null("data.publisher"), col_or_null("data.developer")).cast("string"))
    .withColumn("genre_raw", col_or_null("data.genre").cast("string"))
    .withColumn("languages_raw", col_or_null("data.languages").cast("string"))
    .withColumn("owners_raw", col_or_null("data.owners").cast("string"))
    .withColumn("discount_raw", col_or_null("data.discount"))
    .withColumn("price_raw", F.coalesce(col_or_null("data.price"), col_or_null("data.finalprice"), col_or_null("data.initialprice")))
    .withColumn("initial_price_raw", col_or_null("data.initialprice"))
    .withColumn("windows", col_or_null("data.platforms.windows").cast("int"))
    .withColumn("mac", col_or_null("data.platforms.mac").cast("int"))
    .withColumn("linux", col_or_null("data.platforms.linux").cast("int"))
    .withColumn("positive", col_or_null("data.positive").cast("long"))
    .withColumn("negative", col_or_null("data.negative").cast("long"))
    .withColumn("ccu", col_or_null("data.ccu").cast("long"))
    .withColumn("categories", col_or_null("data.categories"))
    .withColumn("release_date_raw", F.coalesce(col_or_null("data.release_date"), col_or_null("data.releaseDate"), col_or_null("data.releasedate")).cast("string"))
)

games = (
    games
    .withColumn("price_num", to_double_from_any(F.col("price_raw")))
    .withColumn("initial_price_num", to_double_from_any(F.col("initial_price_raw")))
    .withColumn("price_eur", F.when(F.col("price_num") > 1000, F.col("price_num")/100).otherwise(F.col("price_num")))
    .withColumn("initial_price_eur", F.when(F.col("initial_price_num") > 1000, F.col("initial_price_num")/100).otherwise(F.col("initial_price_num")))
    .withColumn("discount_pct", pct_from_any(F.col("discount_raw")))
)

games = (
    games
    .withColumn("n_reviews", (F.coalesce(F.col("positive"), F.lit(0)) + F.coalesce(F.col("negative"), F.lit(0))).cast("long"))
    .withColumn("positive_rate", F.when(F.col("n_reviews") > 0, F.col("positive")/F.col("n_reviews")).otherwise(F.lit(None)).cast("double"))
    .withColumn("popularity_score", (F.log1p(F.col("n_reviews")) * F.col("positive_rate")).cast("double"))
    .withColumn("nb_platforms", (F.coalesce(F.col("windows"),F.lit(0)) + F.coalesce(F.col("mac"),F.lit(0)) + F.coalesce(F.col("linux"),F.lit(0))).cast("int"))
)

games = games.withColumn(
    "release_date",
    F.coalesce(
        F.to_date("release_date_raw", "yyyy-MM-dd"),
        F.to_date("release_date_raw", "MMM d, yyyy"),
        F.to_date("release_date_raw", "d MMM, yyyy"),
        F.to_date("release_date_raw", "yyyy")
    )
).withColumn("release_year", F.year("release_date"))

display(games.select(
    "app_id","name","publisher","developer","release_date_raw","release_year",
    "price_eur","discount_pct","n_reviews","positive_rate","popularity_score",
    "windows","mac","linux","nb_platforms"
).limit(20))

print("games rows =", games.count())


## Normalize nested/text fields (genres, languages, categories)

In [ ]:
games_genres = (
    games
    .withColumn("genre_arr", F.split(F.coalesce(F.col("genre_raw"), F.lit("")), r",\s*"))
    .withColumn("genre", F.explode_outer("genre_arr"))
    .withColumn("genre", F.trim(F.lower(F.col("genre"))))
    .filter(F.col("genre").isNotNull() & (F.col("genre") != ""))
    .select("app_id","publisher","genre","price_eur","discount_pct","n_reviews","positive_rate","popularity_score","nb_platforms")
)
display(games_genres.limit(20))

games_languages = (
    games
    .withColumn("lang_arr", F.split(F.coalesce(F.col("languages_raw"), F.lit("")), r",\s*"))
    .withColumn("language", F.explode_outer("lang_arr"))
    .withColumn("language", F.trim(F.lower(F.col("language"))))
    .filter(F.col("language").isNotNull() & (F.col("language") != ""))
    .select("app_id","language")
)
display(games_languages.limit(20))

games_categories = (
    games
    .withColumn("category", F.explode_outer("categories"))
    .withColumn("category", F.trim(F.lower(F.col("category").cast("string"))))
    .filter(F.col("category").isNotNull() & (F.col("category") != ""))
    .select("app_id","category")
)
display(games_categories.limit(20))


## Macro dashboard tables

In [ ]:
publisher_most_games = (
    games.groupBy("publisher")
    .agg(F.countDistinct("app_id").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
)
display(publisher_most_games.limit(50))

releases_by_year = (
    games.filter(F.col("release_year").isNotNull())
    .groupBy("release_year")
    .agg(F.countDistinct("app_id").alias("nb_releases"))
    .orderBy("release_year")
)
display(releases_by_year)

price_dist = games.select("price_eur").filter(F.col("price_eur").isNotNull())
display(price_dist)

discount_dist = games.select("discount_pct").filter(F.col("discount_pct").isNotNull())
display(discount_dist)

top_languages = (
    games_languages.groupBy("language")
    .agg(F.countDistinct("app_id").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
)
display(top_languages.limit(50))


## Best games (rating proxy)

In [ ]:
best_games = (
    games
    .filter(F.col("positive_rate").isNotNull() & (F.col("n_reviews") >= 500))
    .select("app_id","name","publisher","price_eur","n_reviews","positive_rate","popularity_score")
    .orderBy(F.desc("positive_rate"), F.desc("n_reviews"))
)
display(best_games.limit(50))


## Genre analysis tables

In [ ]:
top_genres = (
    games_genres.groupBy("genre")
    .agg(F.countDistinct("app_id").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
)
display(top_genres.limit(50))

genre_performance = (
    games_genres.groupBy("genre")
    .agg(
        F.countDistinct("app_id").alias("nb_games"),
        F.avg("positive_rate").alias("avg_positive_rate"),
        F.avg("popularity_score").alias("avg_popularity_score"),
        F.avg("price_eur").alias("avg_price_eur"),
        F.avg("discount_pct").alias("avg_discount_pct")
    )
    .filter(F.col("nb_games") >= 200)
    .orderBy(F.desc("avg_popularity_score"))
)
display(genre_performance.limit(50))


## Platform analysis tables

In [ ]:
platform_share = games.agg(
    F.sum(F.coalesce(F.col("windows"), F.lit(0))).alias("nb_windows"),
    F.sum(F.coalesce(F.col("mac"), F.lit(0))).alias("nb_mac"),
    F.sum(F.coalesce(F.col("linux"), F.lit(0))).alias("nb_linux")
)
display(platform_share)

multiplatform_vs_popularity = (
    games.groupBy("nb_platforms")
    .agg(
        F.countDistinct("app_id").alias("nb_games"),
        F.avg("popularity_score").alias("avg_popularity_score"),
        F.avg("positive_rate").alias("avg_positive_rate"),
        F.avg("n_reviews").alias("avg_n_reviews")
    )
    .orderBy("nb_platforms")
)
display(multiplatform_vs_popularity)


## Optional: save processed tables (DBFS)

In [ ]:
# OUT = "dbfs:/FileStore/steam/processed"
# games.write.mode("overwrite").parquet(f"{OUT}/games")
# games_genres.write.mode("overwrite").parquet(f"{OUT}/games_genres")
# games_languages.write.mode("overwrite").parquet(f"{OUT}/games_languages")
# games_categories.write.mode("overwrite").parquet(f"{OUT}/games_categories")
# print("Saved to:", OUT)
